# Fine-tune NusaBERT-large on NusaX-Senti

Fine-tune NusaBERT-large untuk sentiment classification pada 4 bahasa target:
- Javanese (jav)
- Sundanese (sun)
- Acehnese (ace)
- Banjarese (bjn)

Berdasarkan script dari [LazarusNLP/NusaBERT](https://github.com/LazarusNLP/NusaBERT)

Tujuan: mendapatkan sentiment classifier per bahasa yang akan digunakan sebagai
tool di Sentiment Validator Agent.

In [1]:
# Install dependencies
# !pip install transformers datasets evaluate accelerate scikit-learn

In [1]:
import evaluate
import numpy as np
from datasets import load_dataset, ClassLabel
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

y:\Michh\Python\Projects\MAGenerator\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
# Clear GPU memory dari run sebelumnya
import torch
import gc

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    print(f"VRAM after clear: {torch.cuda.memory_allocated()/1024**2:.0f} MB allocated")
    print(f"VRAM reserved: {torch.cuda.memory_reserved()/1024**2:.0f} MB reserved")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

VRAM after clear: 0 MB allocated
VRAM reserved: 0 MB reserved
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [ ]:
# Konfigurasi — NusaBERT-large hyperparams sesuai paper Table 4 + GitHub run_classification.sh
MODEL_CHECKPOINT = "LazarusNLP/NusaBERT-large"
DATASET_NAME = "indonlp/NusaX-senti"
# TARGET_LANGS = ["jav", "sun", "ace", "bjn"]
TARGET_LANGS = ["ace", "ban", "bbc", "bjn", "bug", "eng", "ind", "jav", "mad", "min", "nij", "sun"]
INPUT_MAX_LENGTH = 128

# Hyperparameters (dari paper + GitHub, NusaBERT-large)
NUM_TRAIN_EPOCHS = 100
LEARNING_RATE = 2e-5        # large = 2e-5 (base = 1e-5)
WEIGHT_DECAY = 0.01
TRAIN_BATCH_SIZE = 16        # paper = 16, -> VRAM 8GB tidak cukup → 4 + grad_accum 4
EVAL_BATCH_SIZE = 64      # paper = 64, turunkan untuk VRAM
# GRADIENT_ACCUMULATION_STEPS = 4  # effective batch = 4 * 4 = 16 (sama dengan paper)
EARLY_STOPPING_PATIENCE = 5

# Random seed untuk reproducibility
SEED = 42

OUTPUT_BASE_DIR = "../outputs/nusabert-sentiment"

In [ ]:
def finetune_nusabert(lang_code: str):
    """
    Fine-tune NusaBERT pada NusaX-Senti untuk satu bahasa.
    Berdasarkan: https://github.com/LazarusNLP/NusaBERT/blob/main/scripts/run_classification.py
    """
    model_name = MODEL_CHECKPOINT.split("/")[-1].lower()
    print(f"\n{'='*50}")
    print(f"Fine-tuning {model_name} on NusaX-Senti [{lang_code}]")
    print(f"{'='*50}")
    
    from datasets import Dataset, DatasetDict, Features, Value, ClassLabel as CL
    from transformers import AutoConfig, BertForSequenceClassification, BertTokenizerFast
    import pandas as pd
    
    data_dir = f"../data/nusax_senti/{lang_code}"
    train_df = pd.read_csv(f"{data_dir}/train.csv")
    valid_df = pd.read_csv(f"{data_dir}/valid.csv")
    test_df = pd.read_csv(f"{data_dir}/test.csv")
    
    label_list = sorted(train_df["label"].unique().tolist())
    label2id = {v: i for i, v in enumerate(label_list)}
    id2label = {i: v for i, v in enumerate(label_list)}
    num_labels = len(label_list)
    print(f"Labels: {label_list} → {label2id}")
    
    for df in [train_df, valid_df, test_df]:
        df["label"] = df["label"].map(label2id)
    
    features = Features({
        "id": Value("int64"),
        "text": Value("string"),
        "label": CL(names=label_list),
    })
    
    dataset = DatasetDict({
        "train": Dataset.from_pandas(train_df, features=features, preserve_index=False),
        "validation": Dataset.from_pandas(valid_df, features=features, preserve_index=False),
        "test": Dataset.from_pandas(test_df, features=features, preserve_index=False),
    })
    print(f"Train: {len(dataset['train'])}, Valid: {len(dataset['validation'])}, Test: {len(dataset['test'])}")
    
    tokenizer = BertTokenizerFast.from_pretrained(MODEL_CHECKPOINT)
    config = AutoConfig.from_pretrained(MODEL_CHECKPOINT)
    config.num_labels = num_labels
    config.label2id = label2id
    config.id2label = id2label
    
    model = BertForSequenceClassification.from_pretrained(
        MODEL_CHECKPOINT, config=config, ignore_mismatched_sizes=True,
    )
    total_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"Model: {model_name} ({total_params:.0f}M params)")
    
    def preprocess_function(examples):
        return tokenizer(examples["text"], max_length=INPUT_MAX_LENGTH, truncation=True)
    
    tokenized_dataset = dataset.map(preprocess_function, batched=True)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    
    f1_metric = evaluate.load("f1")
    accuracy_metric = evaluate.load("accuracy")
    
    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")
        acc = accuracy_metric.compute(predictions=predictions, references=labels)
        return {"f1": round(f1["f1"], 4), "accuracy": round(acc["accuracy"], 4)}
    
    output_dir = f"{OUTPUT_BASE_DIR}/{model_name}-{lang_code}"
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        # gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        optim="adamw_torch_fused",
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        bf16=torch.cuda.is_available(),
        report_to="none",
        logging_steps=16,
        seed=SEED,
        data_seed=SEED,
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(EARLY_STOPPING_PATIENCE)],
    )
    
    trainer.train()
    
    test_results = trainer.evaluate(tokenized_dataset["test"])
    print(f"\nTest Results [{lang_code}] ({model_name}):")
    print(f"F1 (macro): {test_results['eval_f1']:.4f}")
    print(f"Accuracy:   {test_results['eval_accuracy']:.4f}")
    
    best_model_dir = f"{output_dir}/best"
    trainer.save_model(best_model_dir)
    tokenizer.save_pretrained(best_model_dir)
    print(f"  Saved to: {best_model_dir}")
    
    return test_results

## Fine-tune per Bahasa

Jalankan fine-tuning untuk masing-masing bahasa target.
Setiap bahasa menghasilkan 1 model classifier yang disimpan di `outputs/nusabert-sentiment/`.

Target F1 (dari paper NusaBERT): jav 87.2%, sun 82.7%, ace 81.8%, bjn 86.5%

In [ ]:
# Fine-tune Javanese
results_jav = finetune_nusabert("jav")

In [ ]:
# Fine-tune Sundanese
results_sun = finetune_nusabert("sun")

In [ ]:
# Fine-tune Acehnese
results_ace = finetune_nusabert("ace")

In [ ]:
# Fine-tune Banjarese
results_bjn = finetune_nusabert("bjn")

## Ringkasan Hasil

In [ ]:
# Ringkasan
import pandas as pd

all_results = {
    "jav": results_jav,
    "sun": results_sun,
    "ace": results_ace,
    "bjn": results_bjn,
}

# Paper reference scores
paper_scores = {"jav": 87.2, "sun": 82.7, "ace": 81.8, "bjn": 86.5}

summary = []
for lang, res in all_results.items():
    summary.append({
        "Language": lang,
        "Our F1 (%)": round(res["eval_f1"] * 100, 2),
        "Paper F1 (%)": paper_scores[lang],
        "Our Accuracy (%)": round(res["eval_accuracy"] * 100, 2),
    })

df_summary = pd.DataFrame(summary)
print("NusaBERT-large Fine-tuning Results on NusaX-Senti (test set)")
print("=" * 60)
display(df_summary)
print("\nPaper scores from: LazarusNLP/NusaBERT (arXiv:2403.01817)")

## Test: Predict Sentiment pada Kalimat Baru

Coba predict pada beberapa kalimat buat lihat output confidence.

In [ ]:
from transformers import pipeline

# Load model yang sudah di-fine-tune
lang = "jav"  # ganti sesuai bahasa yang mau ditest
model_path = f"{OUTPUT_BASE_DIR}/nusabert-large-{lang}/best"

classifier = pipeline("sentiment-analysis", model=model_path, tokenizer=model_path)

# Test kalimat
test_sentences = [
    "Panganan e enak tenan, regane mirah",          # positive
    "Pelayanane ala banget, ora profesional",        # negative  
    "Toko iki buka jam 8 esuk sampek jam 10 bengi",  # neutral
]

for sent in test_sentences:
    result = classifier(sent, top_k=3)
    print(f"\nText: {sent}")
    for r in result:
        print(f"  {r['label']}: {r['score']:.4f}")

## Hasil Fine-tuning NusaBERT-large (dari script .py)

Load model yang sudah di-train dari `outputs/nusabert-sentiment/`.
Urutan: Training History → Evaluate Test Set → Test Predict.

In [ ]:
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from transformers import AutoModelForSequenceClassification, BertTokenizerFast, pipeline
import evaluate

MODEL_NAME = "nusabert-large"
ALL_LANGS = ["ace", "ban", "bbc", "bjn", "bug", "eng", "ind", "jav", "mad", "min", "nij", "sun"]
TARGET_LANGS = ["jav", "sun", "ace", "bjn", "mad", "min", "ban"]  # 7 bahasa target
BASE_DIR = Path("../outputs/nusabert-sentiment")
DATA_DIR = Path("../data/nusax_senti")

# Paper reference (NusaBERT-large, Table 6)
paper_scores = {
    "ace": 81.8, "ban": 82.8, "bbc": 74.7, "bjn": 86.5, "bug": 73.4,
    "eng": 84.6, "ind": 93.3, "jav": 87.2, "mad": 82.5, "min": 83.5,
    "nij": 77.7, "sun": 82.7,
}

print(f"Model: {MODEL_NAME}")
print(f"Output dir: {BASE_DIR}")
print(f"Available languages: {[l for l in ALL_LANGS if (BASE_DIR / f'{MODEL_NAME}-{l}' / 'best').exists()]}")

In [ ]:
# 1. Training History Plot (Val F1 per epoch)
langs_with_history = [l for l in ALL_LANGS if (BASE_DIR / f"{MODEL_NAME}-{l}" / "train_history.json").exists()]
n = len(langs_with_history)

if n == 0:
    print("No training history found!")
else:
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = axes.flatten()
    
    for idx, lang in enumerate(langs_with_history):
        with open(BASE_DIR / f"{MODEL_NAME}-{lang}" / "train_history.json") as f:
            history = json.load(f)
        
        # Deduplicate per epoch
        eval_entries = [h for h in history if "eval_f1" in h]
        seen_epochs = {}
        for h in eval_entries:
            seen_epochs[h["epoch"]] = h
        eval_entries = list(seen_epochs.values())
        
        epochs = [h["epoch"] for h in eval_entries]
        f1s = [h["eval_f1"] * 100 for h in eval_entries]
        
        ax = axes[idx]
        ax.plot(epochs, f1s, marker=".", markersize=3, color="tab:blue", linewidth=1.5)
        
        # Best val F1
        best_f1 = max(f1s)
        best_epoch = epochs[f1s.index(best_f1)]
        ax.plot(best_epoch, best_f1, "o", color="green", markersize=7, zorder=5,
                markeredgecolor="darkgreen", markeredgewidth=1,
                label=f"Best: {best_f1:.1f}% (ep {best_epoch:.0f})")
        
        ax.set_title(f"{lang.upper()}", fontsize=10, fontweight="bold")
        ax.set_xlabel("Epoch", fontsize=8)
        ax.set_ylabel("Val F1 (%)", fontsize=8)
        ax.tick_params(labelsize=7)
        ax.set_xlim(epochs[0], epochs[-1])
        ax.legend(loc="lower right", fontsize=7)
        ax.grid(True, alpha=0.2)
    
    for idx in range(n, len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle(f"{MODEL_NAME} — Validation F1 per Epoch", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

In [ ]:
# 2. Evaluate semua model pada test set
f1_metric = evaluate.load("f1")
accuracy_metric = evaluate.load("accuracy")

results = {}
for lang in ALL_LANGS:
    model_path = BASE_DIR / f"{MODEL_NAME}-{lang}" / "best"
    if not model_path.exists():
        continue
    
    clf = pipeline("text-classification", model=str(model_path), tokenizer=str(model_path),
                   device=0 if torch.cuda.is_available() else -1, top_k=None)
    
    test_df = pd.read_csv(DATA_DIR / lang / "test.csv")
    preds_raw = clf(test_df["text"].tolist(), batch_size=64)
    pred_labels = [max(p, key=lambda x: x["score"])["label"] for p in preds_raw]
    
    label_list = sorted(set(test_df["label"]))
    label2id = {v: i for i, v in enumerate(label_list)}
    true_ids = [label2id[l] for l in test_df["label"]]
    pred_ids = [label2id[l] for l in pred_labels]
    
    f1 = f1_metric.compute(predictions=pred_ids, references=true_ids, average="macro")["f1"]
    acc = accuracy_metric.compute(predictions=pred_ids, references=true_ids)["accuracy"]
    results[lang] = {"f1": f1 * 100, "accuracy": acc * 100}
    
    del clf
    torch.cuda.empty_cache()

# Tabel ringkasan
summary = []
for lang in ALL_LANGS:
    if lang not in results:
        continue
    summary.append({
        "Target": "*" if lang in TARGET_LANGS else "",
        "Language": lang,
        "Our F1 (%)": round(results[lang]["f1"], 2),
        "Paper F1 (%)": paper_scores[lang],
        "Diff (%)": round(results[lang]["f1"] - paper_scores[lang], 2),
        "Our Acc (%)": round(results[lang]["accuracy"], 2),
    })

df_summary = pd.DataFrame(summary)
print("NusaBERT-large on NusaX-Senti test set")
print("=" * 65)
display(df_summary)
print("* = target language | Paper: NusaBERT (arXiv:2403.01817)")

## Test Predict

Contoh prediksi pada beberapa kalimat per bahasa.

In [ ]:
# 3. Test predict
test_sentences = {
    "jav": [
        ("Panganan e enak tenan, regane mirah", "positive"),
        ("Pelayanane ala banget, ora profesional", "negative"),
        ("Toko iki buka jam 8 esuk sampek jam 10 bengi", "neutral"),
    ],
    "sun": [
        ("Dahareunna raos pisan, hargina oge mirah", "positive"),
        ("Tempatna kotor jeung teu meresih", "negative"),
        ("Ieu toko aya di jalan raya", "neutral"),
    ],
    "ace": [
        ("Kueh jih mangat that, yum pih murah", "positive"),
        ("Pelayanan jih jai that, hana sopan", "negative"),
        ("Kedai nyoe buka tiep uroe", "neutral"),
    ],
}

for lang, sentences in test_sentences.items():
    model_path = BASE_DIR / f"{MODEL_NAME}-{lang}" / "best"
    if not model_path.exists():
        continue
    
    clf = pipeline("text-classification", model=str(model_path), tokenizer=str(model_path),
                   device=0 if torch.cuda.is_available() else -1, top_k=None)
    
    print(f"\n{lang.upper()}")
    print("-" * 50)
    
    for text, expected in sentences:
        preds = clf(text)[0]
        preds_sorted = sorted(preds, key=lambda x: x["score"], reverse=True)
        predicted = preds_sorted[0]["label"]
        confidence = preds_sorted[0]["score"]
        
        print(f"  Text: {text}")
        print(f"  Expected: {expected} | Predicted: {predicted} ({confidence:.3f})")
        for p in preds_sorted:
            print(f"    {p['label']:>10}: {p['score']:.4f}")
        print()
    
    del clf
    torch.cuda.empty_cache()